# LangChain — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-2/langchain.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-2/langchain.ipynb)

Companion to: [LangChain](https://anacodicai.org/learn/ai-agents/6-2/langchain)


## What you'll build

The **real `langchain` / `langchain-core`** libraries — not a simulation.
We build an LCEL (LangChain Expression Language) chain:

`prompt template -> "LLM" -> output parser`

using the pipe operator (`|`), then swap the SAME chain's model for a
different one without touching the prompt or parser.

**No API key needed** — `FakeListLLM` is a real, built-in LangChain class
that returns pre-programmed responses in order. It IS an `LLM` in every way
LangChain's plumbing cares about; it just doesn't call out to a network.
`pip install langchain langchain-core` if running locally.

### Setup

In [1]:
# %pip install -q langchain langchain-core
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.language_models.fake import FakeListLLM

print("real langchain-core imported")

real langchain-core imported


## Step 1 — a prompt template

`{topic}` is a real template variable — LangChain fills it in at invoke time,
the same as it would for a production prompt.

In [2]:
prompt = ChatPromptTemplate.from_template(
    "Write one sentence explaining {topic} to a beginner."
)
print(prompt.format(topic="gradient descent"))

Human: Write one sentence explaining gradient descent to a beginner.


## Step 2 — a fake LLM that stands in for a real one

`FakeListLLM` returns its `responses` list in order, one per call. Everything
downstream of it — the chain wiring, the parser, the invocation — is
identical to what you'd write against a real `ChatOpenAI` or `ChatAnthropic`.

In [3]:
llm = FakeListLLM(responses=[
    "Gradient descent nudges a model's parameters downhill on the error "
    "surface until the error stops shrinking.",
    "A vector database stores embeddings so that similar meanings end up "
    "near each other, making semantic search possible.",
])
print(type(llm).__name__, "is a real langchain_core LLM subclass")

# a throwaway instance just to show what .invoke() returns, so it doesn't
# consume from the `responses` list `llm` will use in the chain below
_demo = FakeListLLM(responses=["(demo output, not part of the real chain below)"])
print("what invoke() returns:", _demo.invoke("anything"))

FakeListLLM is a real langchain_core LLM subclass
what invoke() returns: (demo output, not part of the real chain below)


## Step 3 — wire the LCEL chain with the pipe operator

This `|` is the actual LangChain `Runnable.__or__` — it's building a real
`RunnableSequence`, not a metaphor for one.

In [4]:
chain = prompt | llm | StrOutputParser()
print(type(chain).__name__)

RunnableSequence


## Step 4 — invoke it twice, consuming the canned responses in order

**Try it:** invoke it a third time — with only 2 responses configured,
`FakeListLLM` raises `StopIteration`. That's a good way to see exactly how
much of "the chain" is really just calling `.invoke()` on whatever object
sits in the `llm` slot.

In [5]:
result_1 = chain.invoke({"topic": "gradient descent"})
print("result 1:", result_1)

result_2 = chain.invoke({"topic": "vector databases"})
print("result 2:", result_2)

assert "gradient" in result_1.lower()
assert "vector" in result_2.lower() or "embedding" in result_2.lower()
print("\nasserts passed: the chain routed prompt -> fake LLM -> parser correctly, twice")

result 1: Gradient descent nudges a model's parameters downhill on the error surface until the error stops shrinking.
result 2: A vector database stores embeddings so that similar meanings end up near each other, making semantic search possible.

asserts passed: the chain routed prompt -> fake LLM -> parser correctly, twice


## Step 5 — swap the model, keep everything else

This is LangChain's actual value proposition: `prompt` and `StrOutputParser()`
are UNCHANGED. Only the middle piece of the pipe changed. If this were a real
deployment, swapping `FakeListLLM(...)` for `ChatOpenAI(...)` or
`ChatAnthropic(...)` would be the only edit needed.

In [6]:
different_llm = FakeListLLM(responses=[
    "Backpropagation computes how much each weight contributed to the "
    "error, so gradient descent knows which direction to nudge it."
])
chain_v2 = prompt | different_llm | StrOutputParser()

result_v2 = chain_v2.invoke({"topic": "gradient descent"})
print("same prompt, same parser, different model:", result_v2)

assert result_v2 != result_1
print("assert passed: the SAME chain shape produced a different real "
      "answer just by swapping the model in the pipe")

same prompt, same parser, different model: Backpropagation computes how much each weight contributed to the error, so gradient descent knows which direction to nudge it.
assert passed: the SAME chain shape produced a different real answer just by swapping the model in the pipe


## Try it yourself

- Replace `StrOutputParser()` with LangChain's `JsonOutputParser` and have
  `FakeListLLM` return a JSON string — see the chain hand back a real parsed
  `dict` instead of a string.
- Add a THIRD stage to the pipe — a plain Python function wrapped in
  `RunnableLambda` — that post-processes the string (e.g., truncates it to
  one clause). Notice it slots into the `|` chain exactly like the LLM did.
- Swap `FakeListLLM` for `langchain_openai.ChatOpenAI(model="gpt-4o-mini")`
  with a real key and re-run Step 4 unmodified — that's the whole point of
  the abstraction.